# Semana 7: Vistas, Procedimientos y Triggers
## Notebook Conceptual (NB1) – Simulación en Python

### Propósito de la sesión
Programar lógica dentro de la base de datos. En este notebook, simularemos el comportamiento de **vistas** y **triggers** utilizando Python y pandas, para entender su funcionamiento antes de implementarlos en un motor SQL real.

### Objetivos de aprendizaje
*   Simular una **vista** como una función que consulta y transforma un DataFrame.
*   Simular un **trigger** como una función que se ejecuta automáticamente al modificar datos.
*   Entender cómo las vistas simplifican consultas complejas.
*   Comprender cómo los triggers mantienen la integridad y auditoría de los datos.
*   Conectar estos conceptos con su implementación en SQL.

## Configuración Inicial

Importamos las librerías necesarias: pandas para manipulación de datos, numpy y datetime.

In [ ]:
# Importamos librerías
import pandas as pd
import numpy as np
from datetime import datetime

# Configuración de pandas para mostrar más columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Fijamos semilla para reproducibilidad
np.random.seed(42)

print("🎯 Librerías importadas correctamente.")

---
## 1. Creación de Datos de Ejemplo

Simularemos una base de datos de ventas con tres tablas: **clientes**, **productos**, **pedidos** y **detalle_pedido**.

In [ ]:
# Tabla clientes
clientes = pd.DataFrame({
    'id_cliente': [1, 2, 3, 4],
    'nombre': ['Juan Pérez', 'María García', 'Carlos López', 'Ana Martínez'],
    'email': ['juan@email.com', 'maria@email.com', 'carlos@email.com', 'ana@email.com'],
    'ciudad': ['Madrid', 'Barcelona', 'Valencia', 'Sevilla']
})

# Tabla productos
productos = pd.DataFrame({
    'id_producto': [101, 102, 103, 104, 105],
    'nombre': ['Laptop', 'Mouse', 'Teclado', 'Monitor', 'Auriculares'],
    'precio': [1200.00, 25.50, 45.00, 300.00, 80.00],
    'stock': [10, 50, 30, 15, 25]
})

# Tabla pedidos (cabecera)
pedidos = pd.DataFrame({
    'id_pedido': [1001, 1002, 1003, 1004, 1005],
    'id_cliente': [1, 2, 1, 3, 1],
    'fecha': ['2025-03-01', '2025-03-02', '2025-03-02', '2025-03-03', '2025-03-04'],
    'total': [0, 0, 0, 0, 0]  # se calculará después
})

# Tabla detalle_pedido
detalle_pedido = pd.DataFrame({
    'id_pedido': [1001, 1001, 1002, 1003, 1003, 1004, 1005],
    'id_producto': [101, 102, 103, 104, 105, 101, 103],
    'cantidad': [1, 2, 1, 1, 1, 1, 2]
})

print("🔷 Tablas creadas:")
print("\nClientes:")
display(clientes)
print("\nProductos:")
display(productos)
print("\nPedidos:")
display(pedidos)
print("\nDetalle Pedido:")
display(detalle_pedido)

---
## 2. Simulación de una Vista

En SQL, una **vista** es una tabla virtual basada en una consulta. En Python, podemos simularla como una **función** que devuelve un DataFrame transformado.

### 2.1. Vista: Pedidos con información de cliente y total

In [ ]:
def vista_pedidos_completa():
    """
    Vista que muestra los pedidos con nombre del cliente, fecha,
    productos, cantidades y calcula el total por línea.
    """
    # Combinar detalle con productos para obtener precio
    df = detalle_pedido.merge(productos, on='id_producto')
    # Calcular subtotal por línea
    df['subtotal'] = df['cantidad'] * df['precio']
    # Agregar información del pedido y cliente
    df = df.merge(pedidos[['id_pedido', 'id_cliente', 'fecha']], on='id_pedido')
    df = df.merge(clientes[['id_cliente', 'nombre', 'ciudad']], on='id_cliente')
    # Seleccionar columnas relevantes
    return df[['id_pedido', 'fecha', 'nombre', 'ciudad', 'id_producto', 'nombre_producto', 'cantidad', 'precio', 'subtotal']]

# Llamamos a la vista
vista = vista_pedidos_completa()
print("🔷 Vista: Pedidos completos con detalles")
display(vista)

### 2.2. Vista: Resumen por cliente (total gastado)

Vista que muestra el total gastado por cada cliente.

In [ ]:
def vista_resumen_clientes():
    """
    Vista que calcula el total gastado por cliente.
    """
    # Calcular subtotales por línea
    df = detalle_pedido.merge(productos, on='id_producto')
    df['subtotal'] = df['cantidad'] * df['precio']
    # Agrupar por pedido para obtener total por pedido
    total_por_pedido = df.groupby('id_pedido')['subtotal'].sum().reset_index()
    total_por_pedido.columns = ['id_pedido', 'total_pedido']
    # Unir con pedidos para obtener cliente
    df2 = total_por_pedido.merge(pedidos[['id_pedido', 'id_cliente']], on='id_pedido')
    # Agrupar por cliente
    resumen = df2.groupby('id_cliente')['total_pedido'].sum().reset_index()
    resumen = resumen.merge(clientes[['id_cliente', 'nombre', 'ciudad']], on='id_cliente')
    return resumen[['id_cliente', 'nombre', 'ciudad', 'total_pedido']]

vista_clientes = vista_resumen_clientes()
print("🔷 Vista: Resumen de gastos por cliente")
display(vista_clientes)

### 2.3. Ventajas de las vistas

*   **Simplicidad**: Los usuarios pueden consultar `vista_pedidos_completa()` sin necesidad de escribir joins complejos.
*   **Seguridad**: Podríamos crear una vista que oculte información sensible (ej. sin mostrar precios).
*   **Consistencia**: Todos usan la misma lógica de negocio.

---
## 3. Simulación de un Trigger

En SQL, un **trigger** es un bloque de código que se ejecuta automáticamente ante un evento (INSERT, UPDATE, DELETE). Simularemos dos triggers:

1.  **Trigger de actualización de stock**: Cuando se inserta una línea de pedido, se reduce el stock del producto.
2.  **Trigger de cálculo de total**: Cuando se inserta o modifica un detalle, se actualiza el total del pedido.

Para simular triggers en Python, usaremos funciones que se llaman **manualmente** después de modificar los datos, imitando el comportamiento automático.

### 3.1. Trigger de actualización de stock

Cada vez que se inserta una línea en `detalle_pedido`, debemos verificar stock suficiente y actualizar.

In [ ]:
# Función que simula el trigger BEFORE INSERT
def trigger_actualizar_stock(nuevo_detalle):
    """
    Verifica stock y actualiza productos.
    nuevo_detalle: diccionario con 'id_producto', 'cantidad'
    """
    global productos  # para modificar el DataFrame global
    
    id_prod = nuevo_detalle['id_producto']
    cantidad = nuevo_detalle['cantidad']
    
    # Verificar stock actual
    stock_actual = productos.loc[productos['id_producto'] == id_prod, 'stock'].values[0]
    
    if stock_actual < cantidad:
        raise ValueError(f"Stock insuficiente para producto {id_prod}. Stock: {stock_actual}, requerido: {cantidad}")
    
    # Actualizar stock
    productos.loc[productos['id_producto'] == id_prod, 'stock'] -= cantidad
    print(f"✅ Stock actualizado: producto {id_prod} nuevo stock: {productos.loc[productos['id_producto'] == id_prod, 'stock'].values[0]}")
    
    return True

# Función para insertar un nuevo detalle (simula INSERT)
def insertar_detalle(id_pedido, id_producto, cantidad):
    global detalle_pedido
    
    nuevo = {'id_pedido': id_pedido, 'id_producto': id_producto, 'cantidad': cantidad}
    
    # Disparar trigger BEFORE INSERT
    try:
        trigger_actualizar_stock(nuevo)
    except ValueError as e:
        print(f"❌ {e}")
        return False
    
    # Insertar el detalle (simulado con append)
    detalle_pedido = pd.concat([detalle_pedido, pd.DataFrame([nuevo])], ignore_index=True)
    print(f"✅ Detalle insertado: {nuevo}")
    return True

In [ ]:
# Mostrar stock inicial
print("🔷 Stock inicial:")
display(productos)

# Insertar un detalle válido
insertar_detalle(1005, 102, 5)  # 5 unidades de Mouse (id 102)

print("\n🔷 Stock después de inserción válida:")
display(productos)

# Intentar insertar con stock insuficiente
insertar_detalle(1005, 101, 20)  # Laptop stock era 10

print("\n🔷 Detalle pedido actual:")
display(detalle_pedido)

### 3.2. Trigger de cálculo de total del pedido

Cada vez que se inserta o modifica un detalle, debemos recalcular el total del pedido y actualizar la tabla `pedidos`.

In [ ]:
# Función que simula trigger AFTER INSERT/UPDATE/DELETE
def trigger_actualizar_total_pedido(id_pedido):
    global pedidos
    
    # Calcular total del pedido sumando cantidad * precio
    df = detalle_pedido[detalle_pedido['id_pedido'] == id_pedido].merge(productos, on='id_producto')
    total = (df['cantidad'] * df['precio']).sum()
    
    # Actualizar la tabla pedidos
    pedidos.loc[pedidos['id_pedido'] == id_pedido, 'total'] = total
    print(f"✅ Total del pedido {id_pedido} actualizado a {total:.2f}")

# Modificamos insertar_detalle para que después de insertar, dispare el trigger de total
def insertar_detalle_con_trigger(id_pedido, id_producto, cantidad):
    if insertar_detalle(id_pedido, id_producto, cantidad):
        trigger_actualizar_total_pedido(id_pedido)
        return True
    return False

In [ ]:
# Mostrar pedidos antes
print("🔷 Pedidos antes:")
display(pedidos)

# Insertar nuevo detalle en pedido 1005
insertar_detalle_con_trigger(1005, 104, 2)  # 2 monitores

print("\n🔷 Pedidos después:")
display(pedidos)

### 3.3. Trigger de auditoría

Simulamos un trigger que registra en una tabla de auditoría cada modificación en `productos` (precio, stock).

In [ ]:
# Tabla de auditoría
auditoria_productos = pd.DataFrame(columns=['id_producto', 'campo', 'valor_anterior', 'valor_nuevo', 'fecha', 'usuario'])

def trigger_auditoria_productos(id_producto, campo, valor_anterior, valor_nuevo):
    global auditoria_productos
    nuevo_registro = pd.DataFrame([{
        'id_producto': id_producto,
        'campo': campo,
        'valor_anterior': valor_anterior,
        'valor_nuevo': valor_nuevo,
        'fecha': datetime.now(),
        'usuario': 'simulado'
    }])
    auditoria_productos = pd.concat([auditoria_productos, nuevo_registro], ignore_index=True)
    print(f"✅ Auditoría registrada: {campo} de producto {id_producto} cambiado de {valor_anterior} a {valor_nuevo}")

# Función para actualizar producto (simula UPDATE)
def actualizar_producto(id_producto, campo, nuevo_valor):
    global productos
    idx = productos[productos['id_producto'] == id_producto].index[0]
    valor_anterior = productos.at[idx, campo]
    
    # Disparar trigger antes de actualizar (podría ser BEFORE o AFTER)
    productos.at[idx, campo] = nuevo_valor
    
    # Trigger AFTER UPDATE
    trigger_auditoria_productos(id_producto, campo, valor_anterior, nuevo_valor)

# Probamos
actualizar_producto(101, 'precio', 1150.00)
actualizar_producto(102, 'stock', 40)

print("\n🔷 Auditoría:")
display(auditoria_productos)

---
## 4. Comparación con SQL

A continuación, mostramos cómo se verían estos mismos ejemplos en SQL (PostgreSQL).

In [ ]:
# Vista en SQL
sql_vista = """
CREATE VIEW vista_pedidos_completa AS
SELECT 
    p.id_pedido, p.fecha,
    c.nombre, c.ciudad,
    pr.id_producto, pr.nombre AS nombre_producto,
    dp.cantidad, pr.precio, (dp.cantidad * pr.precio) AS subtotal
FROM pedidos p
JOIN clientes c ON p.id_cliente = c.id_cliente
JOIN detalle_pedido dp ON p.id_pedido = dp.id_pedido
JOIN productos pr ON dp.id_producto = pr.id_producto;
"""

# Trigger de actualización de stock en PostgreSQL
sql_trigger_stock = """
CREATE OR REPLACE FUNCTION actualizar_stock()
RETURNS TRIGGER AS $$
BEGIN
    IF (SELECT stock FROM productos WHERE id_producto = NEW.id_producto) < NEW.cantidad THEN
        RAISE EXCEPTION 'Stock insuficiente';
    END IF;
    UPDATE productos SET stock = stock - NEW.cantidad
    WHERE id_producto = NEW.id_producto;
    RETURN NEW;
END;
$$ LANGUAGE plpgsql;

CREATE TRIGGER tr_actualizar_stock
BEFORE INSERT ON detalle_pedido
FOR EACH ROW EXECUTE FUNCTION actualizar_stock();
"""

print("✅ Código SQL equivalente (para referencia)")

---
## Ejercicios para el Estudiante

1.  **Vista de productos con stock bajo**: Crea una función `vista_stock_bajo(umbral)` que devuelva los productos con stock menor al umbral.

2.  **Trigger de validación**: Simula un trigger que impida insertar un pedido si el cliente no existe en la tabla clientes. Modifica `insertar_detalle` para incluir esta validación.

3.  **Trigger de fecha automática**: Al insertar un pedido, asigna automáticamente la fecha actual si no se proporciona.

4.  **Vista de pedidos por ciudad**: Crea una vista que muestre el total de ventas por ciudad.

5.  **Reflexión**: ¿Qué ventajas tiene implementar esta lógica directamente en la base de datos (con triggers y vistas) en lugar de hacerlo en la aplicación? ¿Qué desventajas?

In [ ]:
# Espacio para resolver ejercicios
# Ejercicio 1: Vista de stock bajo
def vista_stock_bajo(umbral=20):
    return productos[productos['stock'] < umbral]

print("Productos con stock < 20:")
display(vista_stock_bajo(20))

---
## Conclusiones de la Sesión

*   Las **vistas** son consultas almacenadas que simplifican el acceso a datos complejos. En Python, las simulamos con funciones que devuelven DataFrames.
*   Los **triggers** ejecutan lógica automáticamente ante eventos de modificación. Simulamos triggers de stock, total de pedido y auditoría.
*   Estas herramientas permiten centralizar reglas de negocio en la base de datos, mejorando consistencia y seguridad.
*   La simulación en Python ayuda a comprender el flujo antes de implementar en un motor real.
*   En la próxima sesión, implementaremos estos conceptos en PostgreSQL usando triggers reales.

¡Ahora entiendes cómo programar lógica dentro de la base de datos!